> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 5 — Bounded Scientific Workflow Agent (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2005.%20Building%20Personal%20Assistants/LC4LSH_Chapter_5_Bounded_Scientific_Workflow.ipynb)

**Learning objectives**
- Define explicit step/cost/tool budgets for an agent
- Enforce budgets with a guardrail wrapper
- Combine retrieval + computation under a bounded loop
- Log every action for auditability

> Runtime: ~10 min (API)  
> Cost: paid LLM required  
> Data: synthetic research task


## Environment setup


### Secrets (Colab or local)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret("LC4LSH_ANTHROPIC_API_KEY", "sk-ant-...")
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")
print("API keys loaded for", API_KEY_PROVIDER)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" "langgraph>=0.2" "pydantic>=2.5" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter5-bounded-workflow"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")


## Why bounded workflows?

An unbounded agent can loop forever, call tools wastefully, and run up cost. A **bounded workflow** wraps the agent with hard limits:

- **Max steps** — stop after N reasoning/tool iterations
- **Max tool calls** — cap external calls
- **Max tokens / cost** — abort when a budget is exceeded
- **Allowed tools** — an allowlist the agent cannot exceed

This is essential for production scientific assistants where reproducibility and cost control matter.


## 1. Define a budget dataclass


In [ ]:
from dataclasses import dataclass, field

@dataclass
class Budget:
    max_steps: int = 6
    max_tool_calls: int = 4
    max_tokens: int = 4000
    allowed_tools: tuple = ("search_kb", "calculator")
    steps: int = 0
    tool_calls: int = 0
    tokens: int = 0
    log: list = field(default_factory=list)

    def record(self, kind, detail, tokens=0):
        self.log.append((kind, detail))
        self.tokens += tokens
        if kind == "step":
            self.steps += 1
        if kind == "tool":
            self.tool_calls += 1

    def exhausted(self):
        return (self.steps >= self.max_steps or
                self.tool_calls >= self.max_tool_calls or
                self.tokens >= self.max_tokens)

budget = Budget()
print("Budget ready:", budget.max_steps, "steps,", budget.max_tool_calls, "tool calls,", budget.max_tokens, "tokens")


## 2. Define allowlisted tools


In [ ]:
from langchain_core.tools import tool

KB = {
    "metformin": "Metformin is a first-line biguanide for type 2 diabetes; it lowers hepatic glucose production.",
    "crispr": "CRISPR-Cas9 is a genome-editing system using a guide RNA to direct Cas9 to a target locus.",
}

@tool("search_kb")
def search_kb(term: str) -> str:
    """Look up a scientific term in a small knowledge base."""
    term = term.lower()
    for k, v in KB.items():
        if k in term:
            return v
    return "No entry found."

@tool("calculator")
def calculator(expr: str) -> str:
    """Evaluate a simple arithmetic expression."""
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

print("Tools:", [t.name for t in (search_kb, calculator)])


## 3. A bounded agent loop


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

TOOLS = {t.name: t for t in (search_kb, calculator)}

def bounded_run(question, budget):
    """A minimal bounded loop: the LLM may call allowlisted tools until budget is exhausted."""
    while not budget.exhausted():
        budget.record("step", "reason")
        prompt = (f"Question: {question}
"
                  f"You may call one of {budget.allowed_tools} by replying TOOL:name:input, "
                  f"or reply FINAL:answer when done.")
        resp = llm.invoke(prompt).content
        budget.record("token", "llm", tokens=len(resp) // 4)
        if resp.startswith("FINAL:"):
            return resp[len("FINAL:"):].strip()
        if resp.startswith("TOOL:"):
            _, name, arg = resp.split(":", 2)
            if name not in budget.allowed_tools:
                budget.record("tool", f"DENIED {name}")
                return f"Aborted: tool '{name}' not in allowlist."
            budget.record("tool", name)
            out = TOOLS[name].invoke(arg.strip())
            question += f"
Tool {name} returned: {out}"
        else:
            return resp
    return "Stopped: budget exhausted. Partial log available."

answer = bounded_run("What is metformin and what is 12*4?", budget)
print("ANSWER:", answer)
print("steps", budget.steps, "| tool_calls", budget.tool_calls, "| tokens~", budget.tokens)


## 4. Inspect the audit log


In [ ]:
for kind, detail in budget.log:
    print(f"{kind:6} | {detail}")


## Limitations & safety notes

- The `calculator` uses `eval` with builtins disabled — still avoid untrusted expressions in production; use a real parser.
- Token accounting here is approximate (`len//4`); use a tokenizer for real budgets.
- This is a didactic loop, not LangGraph; for stateful graphs use LangGraph with explicit check pointers.
- **Paid API required**.


In [ ]:
# Cleanup
import gc
for _v in ("llm", "model", "agent", "graph", "app", "workflow"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why an allowlist of tools?</summary>It prevents the agent from invoking unintended or dangerous tools, a key guardrail.</details>

<details><summary>What happens when the budget is exhausted?</summary>The loop returns a 'budget exhausted' message with a partial log instead of looping forever.</details>

<details><summary>Why log every action?</summary>For auditability and reproducibility — you can replay or debug exactly what the agent did.</details>

### Tasks
- **Task A** - Add a `max_cost_usd` budget that estimates cost from token counts and aborts when exceeded.
- **Task B** - Replace the string-protocol with native tool-calling (`llm.bind_tools`) under the same budget.
- **Task C** - Add a `web_search` tool but keep it off the allowlist; show the agent is denied.
- **Task D** - Persist the audit log to JSON and reload it to reconstruct the run.
